In [0]:

-- Exploração inicial para entender qualidade e distribuição do
-- dado antes de definir as regras de limpeza da camada silver.


-- 1. Volume geral
select count(*) as total_linhas
from workspace.bronze.nyc_taxi_trips;

select * from workspace.bronze.nyc_taxi_trips limit 1;

-- 2. Checagem de nulos por coluna
-- Objetivo: identificar quais colunas justificam filtro de
-- qualidade na camada silver (ver stg_taxi_trips.sql).
select count(*) as nulos_vendorid
from workspace.bronze.nyc_taxi_trips where vendorid is null;

select count(*) as nulos_pickup_datetime
from workspace.bronze.nyc_taxi_trips where tpep_pickup_datetime is null;

select count(*) as nulos_dropoff_datetime
from workspace.bronze.nyc_taxi_trips where tpep_dropoff_datetime is null;

select count(*) as nulos_passenger_count
from workspace.bronze.nyc_taxi_trips where passenger_count is null;

select count(*) as nulos_trip_distance
from workspace.bronze.nyc_taxi_trips where trip_distance is null;

select count(*) as nulos_ratecodeid
from workspace.bronze.nyc_taxi_trips where ratecodeid is null;

select count(*) as nulos_store_and_fwd_flag
from workspace.bronze.nyc_taxi_trips where store_and_fwd_flag is null;

select count(*) as nulos_pulocationid
from workspace.bronze.nyc_taxi_trips where pulocationid is null;

select count(*) as nulos_dolocationid
from workspace.bronze.nyc_taxi_trips where dolocationid is null;

select count(*) as nulos_payment_type
from workspace.bronze.nyc_taxi_trips where payment_type is null;

select count(*) as nulos_fare_amount
from workspace.bronze.nyc_taxi_trips where fare_amount is null;

select count(*) as nulos_extra
from workspace.bronze.nyc_taxi_trips where extra is null;

select count(*) as nulos_mta_tax
from workspace.bronze.nyc_taxi_trips where mta_tax is null;

select count(*) as nulos_tip_amount
from workspace.bronze.nyc_taxi_trips where tip_amount is null;

select count(*) as nulos_tolls_amount
from workspace.bronze.nyc_taxi_trips where tolls_amount is null;

select count(*) as nulos_improvement_surcharge
from workspace.bronze.nyc_taxi_trips where improvement_surcharge is null;


-- Corridas com mais de 4 passageiros (possível erro de digitação
-- do motorista, ou corrida em grupo — ver RatecodeID = 6).
select *
from workspace.bronze.nyc_taxi_trips
where passenger_count > 4;

-- Corridas longas (> 10 km), candidatas a outlier ou corrida
-- interestadual (ex: aeroporto -> fora da cidade).
with km_trip as (
    select
        round(trip_distance * 1.60934, 2) as trip_km,
        *
    from workspace.bronze.nyc_taxi_trips
)
select *
from km_trip
where trip_km > 10;


-- - dropoff nulo ou anterior ao pickup -> excluído
-- - duração > 3h -> excluído (provável erro de sensor/GPS)
-- - dolocationid, payment_type, passenger_count, trip_distance
--   nulos -> excluídos
-- - id de corrida inexistente no dataset original -> gerado via
-- - trip_id a partir de vendor + timestamps + zonas
